<a href="https://colab.research.google.com/github/sabbir179/Multi_Agent_Medical_AI/blob/master/notebooks/dataset_verification_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A Multi-Agent Framework for Medical AI: Leveraging Fine-Tuned GPT, LLaMA, and DeepSeek R1 for Evidence-Based and Bias-Aware Clinical Query Processin — Reproduction & Revision Work
**Original Framework & Research:** Naomi Nourmohammadi, Md Meem Hossain,
The Anh Han, Safina Showkat Ara, Zia Ush Shamszaman


**Supervisor:** Dr. Zia Ush Shamszaman
**Collaborator:** Md Meem Hossain

**This Notebook — Reproduction, Verification & Documentation:** Sabbir Ahmed

**Purpose:** Reproducing the framework for IP&M resubmission (IPM-D-25-04242R1)
following reviewer feedback. Original fine-tuned checkpoints are no longer
accessible (confirmed by original author) — this notebook documents full
reproduction from source code, dataset, and published settings.

## 1. Environment Setup

In [1]:
!git clone https://github.com/sabbir179/Multi_Agent_Medical_AI
%cd Multi_Agent_Medical_AI

Cloning into 'Multi_Agent_Medical_AI'...
remote: Enumerating objects: 118, done.
remote: Counting objects: 100% (118/118), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 118 (delta 51), reused 4 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (118/118), 70.07 KiB | 4.38 MiB/s, done.
Resolving deltas: 100% (51/51), done.
/content/Multi_Agent_Medical_AI


In [2]:
!pip install -r requirements.txt


ERROR: Could not find a version that satisfies the requirement gc (from versions: none)
ERROR: No matching distribution found for gc


In [3]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


True
Tesla T4


## 2. Reviewing Project Documentation
Checking the original README for setup instructions and model links.

In [4]:
!cat README.md


# 🏥 Multi Agent Medical AI 🤖  
🚀 This is an AI-driven **medical question-answering system** integrating **LLaMA 3.1, DeepSeek R1, and Transformer models** to provide **accurate, evidence-based medical responses**.  
It is powered by **3 AI Agents**, **Evidence Retrieval AI Agent**, **Clinical Reasoning AI Agent**, **Uncertainty & Refinement AI Agent**, and features **Human-in-the-Loop (HITL) Review** for ensuring reliability and reducing AI bias.


---
## 🔥 What's Special About MedAI?
✅ **Fine-Tuned LLaMA & DeepSeek on 20,000+ Medical Data Points** 📊  
   - Trained on a **large-scale medical dataset** for **domain-specific accuracy**.  
   - Ensures **high-quality, expert-verified medical answers**.  

📌 **GitHub Link to Fine-Tuned Models:**  
🔗 **[Fine-Tuned LLaMA 3.1 Model](https://github.com/Naominour/Fine_Tuning_LLaMA_Model)**  
🔗 **[Fine-Tuned DeepSeek R1 Model](https://github.com/Naominour/Fine-tuning-DeepSeek-R1)**  

---

## 🤖 AI Agent Architecture (Human-in-the-Loop)
MedAI's r

## 3. Locating the Fine-Tuned Model Checkpoints
Cloning the two training repositories referenced in the README to check
whether trained model weights are available.

In [5]:
%cd /content
!git clone https://github.com/Naominour/Fine_Tuning_LLaMA_Model.git
!git clone https://github.com/Naominour/Fine-tuning-DeepSeek-R1.git

/content
Cloning into 'Fine_Tuning_LLaMA_Model'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 130 (delta 68), reused 46 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (130/130), 248.53 KiB | 5.40 MiB/s, done.
Resolving deltas: 100% (68/68), done.
Cloning into 'Fine-tuning-DeepSeek-R1'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 33 (delta 10), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 258.87 KiB | 7.84 MiB/s, done.
Resolving deltas: 100% (10/10), done.


In [6]:
!ls -lhR /content/Fine_Tuning_LLaMA_Model
!ls -lhR /content/Fine-tuning-DeepSeek-R1

/content/Fine_Tuning_LLaMA_Model:
total 212K
-rw-r--r-- 1 root root 4.2K Nov 23 19:58 app.py
-rw-r--r-- 1 root root  34K Nov 23 19:58 llama31.py
drwxr-xr-x 2 root root 4.0K Nov 23 19:58 llama-models
-rw-r--r-- 1 root root 4.3K Nov 23 19:58 medical_data.py
-rw-r--r-- 1 root root 6.0K Nov 23 19:58 README.md
-rw-r--r-- 1 root root  11K Nov 23 19:58 reference.py
-rw-r--r-- 1 root root 124K Nov 23 19:58 result.ipynb
drwxr-xr-x 2 root root 4.0K Nov 23 19:58 src
-rw-r--r-- 1 root root 5.6K Nov 23 19:58 tokenizer.py

/content/Fine_Tuning_LLaMA_Model/llama-models:
total 0

/content/Fine_Tuning_LLaMA_Model/src:
total 24K
-rw-r--r-- 1 root root   1 Nov 23 19:58 image
-rw-r--r-- 1 root root 20K Nov 23 19:58 result.png
/content/Fine-tuning-DeepSeek-R1:
total 272K
-rw-r--r-- 1 root root 3.6K Nov 23 19:58 DeepSeek-R1.py
-rw-r--r-- 1 root root  99K Nov 23 19:58 plots.png
-rw-r--r-- 1 root root 2.3K Nov 23 19:58 README.md
-rw-r--r-- 1 root root 161K Nov 23 19:58 response.png


## 4. Reviewing the LLaMA Training Script
Checking required data format, hyperparameters, and dependencies.

In [7]:
!cat /content/Fine_Tuning_LLaMA_Model/llama31.py


import os
import glob
import fire
import time
import json
import math
import torch
import mlflow
import numpy as np
from torch import nn
import matplotlib.pyplot as plt
import torch.nn.functional as F

from pathlib import Path
from dataclasses import dataclass
from typing import List, Optional, Tuple, TypedDict
from torch import nn
from tokenizer import Tokenizer
# -----------------------------------------------------------------------------
# ModelArgs


@dataclass
class ModelArgs:
    """
    A dataclass for storing and configuring model hyperparameters.
    """
    dim: int = 4096  # Dimensionality of model embeddings
    n_layers: int = 32  # Number of transformer layers
    n_heads: int = 32  # Number of attention heads
    n_kv_heads: Optional[int] = None  # Number of key-value heads (default: equal to n_heads)
    vocab_size: int = -1  # Size of the vocabulary
    multiple_of: int = 256  # SwiGLU hidden layer size must be a multiple of this value
    ffn_dim_multiplier: Optional

## 5. Reviewing the DeepSeek Training Script
Checking required data format, hyperparameters, and dependencies.

In [8]:
!cat /content/Fine-tuning-DeepSeek-R1/DeepSeek-R1.py

from unsloth import FastLanguageModel
from unsloth import is_bfloat16_supported # Check if the hardware support bfloat16 precision
from trl import SFTTrainer # Supervised Fine-Tuning Trainer
from huggingface_hub import login
from transformers import TrainingArguments # Define training hyperparameter
from datasets import load_dataset

import os
import torch
import wandb




# Initialise Hugging Face and Wandb tokens
huggingface_token = os.environ.get("HF_TOKEN")
wandb_token = os.environ.get("wab")

# Login to Hugginface & Wandb
login(huggingface_token)
wandb.login(key=wandb_token)
run = wandb.init(
    project="Fine-tuning DeepSeek-R1-Distill-Llama-8B on Medical Dataset",
    job_type="training",
    anonymous="allow"
)

# Set parameters
max_seq_length = 2048 # Max sequence length the model can handle
dtype = None
load_in_4bit = True # 4-bit Quantisation

# Load model & tokenizer using Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Di

## 6. Dataset Acquisition
Downloading MedQuAD from the Kaggle source confirmed by the original author.

In [9]:
!pip install kagglehub
import kagglehub
path = kagglehub.dataset_download("pythonafroz/medquad-medical-question-answer-for-ai-research")
print(path)

100%|██████████| 4.95M/4.95M [00:00<00:00, 7.06MB/s]

Extracting files...
/root/.cache/kagglehub/datasets/pythonafroz/medquad-medical-question-answer-for-ai-research/versions/1


In [10]:
!ls -lh {path}


total 22M
-rw-r--r-- 1 root root 22M Nov 23 19:58 medquad.csv


## 7. Dataset Inspection
Checking structure, column names, and row count.

In [11]:
import pandas as pd
df = pd.read_csv(f"{path}/medquad.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

(16412, 4)
['question', 'answer', 'source', 'focus_area']


,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma


## 8. Data Preprocessing
Note: the 80/20 row-level split created here is superseded by Section 11.9.

Renaming columns to match the training script format, removing duplicates,
and creating an 80/20 train/test split (seed=3407) for reproducibility.

In [12]:
from sklearn.model_selection import train_test_split

# Rename columns to match training script
df = df.rename(columns={"question": "Question", "answer": "Answer"})

# Remove duplicates
df = df.drop_duplicates(subset=["Question", "Answer"]).reset_index(drop=True)
print("After dedup:", df.shape)

# Create train/test split (80/20, fixed seed for reproducibility)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=3407)
df.loc[train_df.index, "split"] = "train"
df.loc[test_df.index, "split"] = "test"

print(df["split"].value_counts())

# Save prepared file
df.to_csv("/content/medquad_prepared.csv", index=False)
print("Saved to /content/medquad_prepared.csv")

After dedup: (16364, 4)
split
train    13091
test      3273
Name: count, dtype: int64
Saved to /content/medquad_prepared.csv


## 9. Backing Up Prepared Data to Google Drive

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
!mkdir -p /content/drive/MyDrive/MedicalAI_Revision
!cp /content/medquad_prepared.csv /content/drive/MyDrive/MedicalAI_Revision/
print("Backed up to Drive")

Backed up to Drive


## 10. Dataset Discrepancy Investigation
The paper reports over 20,000 QA pairs; the working dataset has 16,364
after deduplication. This section verifies the dataset against two
independent sources.

### 10.1 Counting the Full Original MedQuAD Dataset (XML source)

In [15]:
%cd /content
!git clone https://github.com/abachaa/MedQuAD.git
!find /content/MedQuAD -name "*.xml" | wc -l

/content
Cloning into 'MedQuAD'...
remote: Enumerating objects: 11310, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 11310 (delta 7), reused 4 (delta 4), pack-reused 11300 (from 1)
Receiving objects: 100% (11310/11310), 11.01 MiB | 17.34 MiB/s, done.
Resolving deltas: 100% (6807/6807), done.
11274


### 10.2 Verifying Against an Independent CSV Conversion

In [16]:
!git clone https://github.com/avery-lockwood/MedQuAD-CSVs.git
!find /content/MedQuAD-CSVs -name "*.csv" | wc -l
!ls -lh /content/MedQuAD-CSVs

Cloning into 'MedQuAD-CSVs'...
remote: Enumerating objects: 11347, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 11347 (delta 8), reused 5 (delta 5), pack-reused 11338 (from 1)
Receiving objects: 100% (11347/11347), 15.98 MiB | 20.40 MiB/s, done.
Resolving deltas: 100% (6822/6822), done.
12
total 984K
-rw-r--r--  1 root root  663 Nov 23 19:59 changes.md
drwxr-xr-x  2 root root 4.0K Nov 23 19:59 _data_as_csvs
drwxr-xr-x 14 root root 4.0K Nov 23 19:59 _data_as_xmls
-rw-r--r--  1 root root  17K Nov 23 19:59 LICENSE.txt
-rw-r--r--  1 root root 938K Nov 23 19:59 QA-TestSet-LiveQA-Med-Qrels-2479-Answers.zip
-rw-r--r--  1 root root 3.0K Nov 23 19:59 readme.txt
-rw-r--r--  1 root root 7.3K Nov 23 19:59 xml_to_csvs.ipynb


### 10.3 Counting QA Pairs Directly from Original XML

In [17]:
import xml.etree.ElementTree as ET
import glob

xml_files = glob.glob("/content/MedQuAD/**/*.xml", recursive=True)
total_pairs = 0
for f in xml_files:
    try:
        tree = ET.parse(f)
        pairs = tree.findall(".//QAPair")
        total_pairs += len(pairs)
    except Exception as e:
        pass

print("Total XML files:", len(xml_files))
print("Total QA pairs:", total_pairs)

Total XML files: 11274
Total QA pairs: 47441


### 10.4 Summing Rows Across the 12 Source CSVs

In [18]:
import pandas as pd
import glob

csv_files = glob.glob("/content/MedQuAD-CSVs/_data_as_csvs/*.csv")
print("CSV files found:", len(csv_files))

total_rows = 0
for f in csv_files:
    df = pd.read_csv(f)
    print(f.split("/")[-1], ":", len(df), "rows")
    total_rows += len(df)

print("\nTOTAL ROWS:", total_rows)

CSV files found: 12
11_MPlusDrugs_QA.csv : 12889 rows
8_NHLBI_QA_XML.csv : 559 rows
7_SeniorHealth_QA.csv : 769 rows
6_NINDS_QA.csv : 1104 rows
9_CDC_QA.csv : 270 rows
1_CancerGov_QA.csv : 729 rows
10_MPlus_ADAM_QA.csv : 17348 rows
2_GARD_QA.csv : 5394 rows
4_MPlus_Health_Topics_QA.csv : 981 rows
3_GHR_QA.csv : 5430 rows
5_NIDDK_QA.csv : 1192 rows
12_MPlusHerbsSupplements_QA.csv : 792 rows

TOTAL ROWS: 47457


### 10.5 Checking for Empty/Missing Answers (CSV source)
Testing whether the size difference is explained by entries with missing
answers, likely due to copyright restrictions on certain NIH sources.

In [20]:
import pandas as pd
import glob

csv_files = glob.glob("/content/MedQuAD-CSVs/_data_as_csvs/*.csv")

empty_answers = 0
valid_answers = 0
for f in csv_files:
    df = pd.read_csv(f)
    empty_answers += df['Answer'].isna().sum() + (df['Answer'].astype(str).str.strip() == "").sum()
    valid_answers += df['Answer'].notna().sum()

print("Rows with empty/missing answers:", empty_answers)
print("Rows with actual answers:", valid_answers)

KeyError: 'Answer'

*Note: initial attempt referenced the column with capitalized/original
naming; corrected below after checking the actual column names in
this data source.*

In [21]:
import pandas as pd
import glob

csv_files = glob.glob("/content/MedQuAD-CSVs/_data_as_csvs/*.csv")

# Check one file's columns first
sample = pd.read_csv(csv_files[0])
print("Columns in this file:", sample.columns.tolist())
print(sample.head(2))

Columns in this file: ['Unnamed: 0', 'question', 'question_id', 'question_type', 'answer']
   Unnamed: 0                                           question question_id  \
0           0  What important warning or information should I...   0000001-1   
1           1  Who should get Abacavir and why is it prescrib...   0000001-2   

       question_type  answer  
0  important warning     NaN  
1         indication     NaN  


In [22]:
import pandas as pd
import glob

csv_files = glob.glob("/content/MedQuAD-CSVs/_data_as_csvs/*.csv")

empty_answers = 0
valid_answers = 0
for f in csv_files:
    df = pd.read_csv(f)
    empty_answers += df['answer'].isna().sum() + (df['answer'].astype(str).str.strip() == "").sum()
    valid_answers += df['answer'].notna().sum() - (df['answer'].astype(str).str.strip() == "").sum()

print("Rows with empty/missing answers:", empty_answers)
print("Rows with actual (non-empty) answers:", valid_answers)
print("Total:", empty_answers + valid_answers)

Rows with empty/missing answers: 31034
Rows with actual (non-empty) answers: 16423
Total: 47457


### 10.6 Verifying Question-Level Content Overlap
Confirming whether every question in the working dataset also exists in
the original MedQuAD source.

In [23]:
import xml.etree.ElementTree as ET
import glob

# Extract actual question texts from original XML
xml_files = glob.glob("/content/MedQuAD/**/*.xml", recursive=True)
xml_questions = set()
for f in xml_files:
    try:
        tree = ET.parse(f)
        for qpair in tree.findall(".//QAPair"):
            q = qpair.find(".//Question")
            if q is not None and q.text:
                xml_questions.add(q.text.strip().lower())
    except Exception:
        pass

print("Unique questions in original XML:", len(xml_questions))

# Compare against Kaggle questions
kaggle_questions = set(df["Question"].str.strip().str.lower())
print("Unique questions in Kaggle file:", len(kaggle_questions))

overlap = kaggle_questions & xml_questions
print("Questions matching between Kaggle and original XML:", len(overlap))
print("Questions in Kaggle NOT found in original XML:", len(kaggle_questions - xml_questions))

Unique questions in original XML: 43449


KeyError: 'Question'

*Note: initial attempt referenced the column with capitalized/original
naming; corrected below after checking the actual column names in
this data source.*

In [24]:
# Reload the prepared Kaggle file to be safe
df_kaggle = pd.read_csv("/content/medquad_prepared.csv")
print("Kaggle columns:", df_kaggle.columns.tolist())

kaggle_questions = set(df_kaggle["Question"].str.strip().str.lower())
print("Unique questions in Kaggle file:", len(kaggle_questions))

overlap = kaggle_questions & xml_questions
print("Questions matching between Kaggle and original XML:", len(overlap))
print("Questions in Kaggle NOT found in original XML:", len(kaggle_questions - xml_questions))

Kaggle columns: ['Question', 'Answer', 'source', 'focus_area', 'split']
Unique questions in Kaggle file: 14349
Questions matching between Kaggle and original XML: 14349
Questions in Kaggle NOT found in original XML: 0


### 10.7 Independent Verification: Empty Answers Counted Directly from XML
Cross-checking Section 10.5's result using the original XML source directly,
rather than the third-party CSV conversion.

In [25]:
empty_in_xml = 0
valid_in_xml = 0
for f in xml_files:
    try:
        tree = ET.parse(f)
        for qpair in tree.findall(".//QAPair"):
            a = qpair.find(".//Answer")
            if a is None or not a.text or not a.text.strip():
                empty_in_xml += 1
            else:
                valid_in_xml += 1
    except Exception:
        pass

print("Empty answers (from original XML directly):", empty_in_xml)
print("Valid answers (from original XML directly):", valid_in_xml)

Empty answers (from original XML directly): 31034
Valid answers (from original XML directly): 16407


## 10.8 Findings Summary

**Verification method:** Content and count comparisons were performed
against two independent sources — the original MedQuAD XML repository
(github.com/abachaa/MedQuAD) and a separate CSV conversion of the same
data (github.com/avery-lockwood/MedQuAD-CSVs) — not counts alone.

**Q1 — Question-level comparison (see Section 10.6):** Question text was compared (stripped, lowercased) between the deduplicated Kaggle file (16,364 pairs, 14,349 unique questions at that stage) and the original MedQuAD XML. All 14,349 unique questions were found in the XML, with zero mismatches. Answer texts were not compared row by row. Five rows with empty answers were identified later and excluded (Sections 11.7 and 11.8).

**Q2 — Empty answers verified from original XML directly (see Sections 10.5 and 10.7):** The CSV conversion gives 31,034 empty and 16,423 non-empty answers; direct XML parsing gives 31,034 empty and 16,407 non-empty. The empty counts match exactly. The non-empty counts differ by 16, as do the total row counts (47,457 vs 47,441). The Kaggle file's non-empty rows (16,407) equal the direct XML count.

**Q3 — Duplicate criteria (see Section 8):** Rows were considered duplicates only when both the Question text and the Answer text were identical (case-sensitive, subset=["Question", "Answer"]); the first occurrence was kept. Repeated questions with different answers were not treated as duplicates.

**Q4 — Kaggle dataset origin (see Section 10.6):** All questions in the Kaggle file are present in the original MedQuAD XML, which indicates it is derived from MedQuAD. Answer text and any preprocessing applied by the uploader were not compared row by row.

**Q5 — Notebook and script:** This notebook is the methodology record. The cleaning and splitting script is clean_medquad.py (GitHub commit c060ff2): https://github.com/sabbir179/Multi_Agent_Medical_AI/blob/master/clean_medquad.py

**Conclusion:** The working dataset (16,359 pairs after excluding 48 exact duplicates and 5 empty-answer rows) corresponds to the answered portion of MedQuAD: the Kaggle file's 16,407 non-empty-answer rows equal the number of non-empty answers counted directly from the original XML (16,407). The paper's stated figure of "20,000+" pairs could not be matched to any verifiable version of MedQuAD, whether full (47,457 or 47,441 pairs), CSV-converted, or answered-only.

## 11. Dataset Finalisation

### 11.1 Summary: Understanding the Dataset Numbers (final)

Several related figures appeared during the investigation. The table and notes below explain exactly how they relate and state which dataset is used for training and evaluation.

| Step | Rows |
|---|---|
| Raw records in the Kaggle file (medquad.csv) | 16,412 |
| Removed: exact duplicate (Question, Answer) pairs | 48 (leaves 16,364) |
| Removed: rows with empty answers (all 5 from GARD) | 5 (leaves 16,359) |
| **Final dataset (medquad_final_v1.csv)** | **16,359** |
| Unique questions in the final dataset (stripped, lowercased) | 14,345 |

**Why 16,364 and 16,359.** 16,364 was the count after duplicate removal only. Five rows with empty answers were found afterwards (Section 11.7) and excluded, giving 16,359, which is the count used from here on.

**Why "around 16,400".** The original MedQuAD contains 47,457 rows in the CSV conversion (47,441 QA pairs when parsed directly from the XML), of which 31,034 have empty answers (Sections 10.5 and 10.7). Counting directly from the original XML gives 16,407 pairs with non-empty answers. The Kaggle file has 16,412 rows, of which 5 have empty answers, leaving 16,407 non-empty rows, the same number. Removing the 48 exact duplicates gives 16,359. The independent CSV conversion gives 16,423 non-empty answers, 16 more than the XML count; its total row count is also 16 higher (47,457 vs 47,441). This is a comparison of counts, not a row-by-row comparison of answers.

**Why unique questions are fewer than pairs.** The 14,345 unique questions are fewer than the 16,359 pairs because 1,396 questions appear more than once, producing 2,014 additional pairs. Most are the same question published by different NIH sources (1,023 of the 1,396 occur in more than one source, e.g. "How many people are affected by Celiac Disease?" in both NIDDK and GHR), each with its own answer text; all 1,396 have more than one distinct answer. Duplicate removal only drops rows where question and answer are both identical (case-sensitive), so these pairs are retained. Because the same question can have several answers, the train/validation/test split is made by unique question rather than by row, so that no question appears in more than one split.

**Inclusion criteria:** all rows of the Kaggle medquad.csv with a non-empty question and a non-empty answer.
**Exclusion criteria:** exact duplicate (Question, Answer) pairs (first occurrence kept); rows with empty answers. No other filtering, reweighting or manual selection.

**Dataset used for training and evaluation:** medquad_final_v1.csv, 16,359 question-answer pairs, SHA-256 c4fe3039215857079bfa815ebe68ce3a71e785c1781c91d8946fb52462376094.

**Splits:** grouped by unique question, 70/15/15, seed 3407. Train 11,404 pairs (10,041 questions), validation 2,476 pairs (2,151 questions), test 2,479 pairs (2,153 questions). Split files and record IDs are saved with SHA-256 hashes in manifest.json.

**Reproducibility:** cleaning script clean_medquad.py (GitHub commit c060ff2), verified to regenerate the dataset and splits with identical hashes (Section 11.10).

### 11.2 Freezing the Final Dataset and Generating a Checksum

Note: this freeze is superseded by Section 11.8 (final dataset v1, 16,359 rows).

Saving the final, deduplicated dataset as the frozen version for all
subsequent experiments, and generating a checksum so this exact file
can be verified later.



In [26]:
import hashlib

# Reload the final cleaned dataset to confirm we're working with the right file
df_final = pd.read_csv("/content/medquad_prepared.csv")
print("Final dataset shape:", df_final.shape)

# Generate a checksum (SHA-256) of the file
def get_file_checksum(filepath):
    sha256_hash = hashlib.sha256()
    with open(filepath, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()

checksum = get_file_checksum("/content/medquad_prepared.csv")
print("SHA-256 checksum:", checksum)

# Save the checksum alongside the dataset for the record
with open("/content/medquad_prepared_CHECKSUM.txt", "w") as f:
    f.write(f"File: medquad_prepared.csv\n")
    f.write(f"Rows: {df_final.shape[0]}\n")
    f.write(f"SHA-256: {checksum}\n")

print("Checksum saved.")

Final dataset shape: (16364, 5)
SHA-256 checksum: 2053d0aa959421da5517ffc00fc7a12ec656625b053949ad2cfbe866d5016395
Checksum saved.


### 11.3 Creating Train, Validation, and Test Splits
Note: this row-level split is superseded by the question-grouped split in Section 11.9.

Splitting the frozen dataset into training, validation, and test sets
using a fixed random seed, and saving the resulting split assignments
so every future experiment uses identical data.


In [27]:
from sklearn.model_selection import train_test_split

# First split: separate out the test set (15%)
train_val_df, test_df = train_test_split(
    df_final, test_size=0.15, random_state=3407
)

# Second split: from the remaining 85%, split into train (70% of total) and validation (15% of total)
train_df, val_df = train_test_split(
    train_val_df, test_size=(0.15/0.85), random_state=3407
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

# Confirm no overlap between splits
train_ids = set(train_df.index)
val_ids = set(val_df.index)
test_ids = set(test_df.index)

print("Train/Val overlap:", len(train_ids & val_ids))
print("Train/Test overlap:", len(train_ids & test_ids))
print("Val/Test overlap:", len(val_ids & test_ids))

# Save each split separately
train_df.to_csv("/content/medquad_train.csv", index=False)
val_df.to_csv("/content/medquad_val.csv", index=False)
test_df.to_csv("/content/medquad_test.csv", index=False)

print("Splits saved.")

Train: (11454, 5)
Validation: (2455, 5)
Test: (2455, 5)
Train/Val overlap: 0
Train/Test overlap: 0
Val/Test overlap: 0
Splits saved.


### 11.4 Backing Up the Frozen Dataset and Splits to Google Drive
Note: the files backed up here are superseded by the frozen_v1 folder (Sections 11.8 to 11.11).

Saving the final dataset, checksum, and train/validation/test split
files to Google Drive to prevent loss if the Colab session disconnects,
and to preserve this exact version for all future experiments.

In [28]:
import shutil

files_to_backup = [
    "/content/medquad_prepared.csv",
    "/content/medquad_prepared_CHECKSUM.txt",
    "/content/medquad_train.csv",
    "/content/medquad_val.csv",
    "/content/medquad_test.csv",
]

for f in files_to_backup:
    shutil.copy(f, "/content/drive/MyDrive/MedicalAI_Revision/")

print("All files backed up to Drive.")

All files backed up to Drive.


### 11.5 Saving the Data Cleaning Script
Note: this script is superseded by clean_medquad.py (Section 11.10).

Saving the preprocessing steps as a standalone script, separate from
this notebook, for the reproducibility record.

In [30]:
cleaning_script = '''
"""
MedQuAD Dataset Cleaning Script
Used to prepare the final training/evaluation dataset for the
Multi-Agent Medical AI project.

Source: Kaggle - pythonafroz/medquad-medical-question-answer-for-ai-research
Original rows: 16,412
Final rows after cleaning: 16,364
"""

import pandas as pd
from sklearn.model_selection import train_test_split

# Load raw data
df = pd.read_csv("medquad.csv")

# Rename columns to match training script format
df = df.rename(columns={"question": "Question", "answer": "Answer"})

# Remove exact duplicate question-answer pairs
df = df.drop_duplicates(subset=["Question", "Answer"]).reset_index(drop=True)

# Create train/validation/test splits (70/15/15), seed=3407
train_val_df, test_df = train_test_split(df, test_size=0.15, random_state=3407)
train_df, val_df = train_test_split(train_val_df, test_size=(0.15/0.85), random_state=3407)

# Save outputs
df.to_csv("medquad_prepared.csv", index=False)
train_df.to_csv("medquad_train.csv", index=False)
val_df.to_csv("medquad_val.csv", index=False)
test_df.to_csv("medquad_test.csv", index=False)
'''

with open("/content/data_cleaning_script.py", "w") as f:
    f.write(cleaning_script)

print("Cleaning script saved.")

# Backup this too
import shutil
shutil.copy("/content/data_cleaning_script.py", "/content/drive/MyDrive/MedicalAI_Revision/")
print("Cleaning script backed up to Drive.")

Cleaning script saved.
Cleaning script backed up to Drive.


### 11.6 Verifying the Dataset Numbers (Reconciliation Counts)
Printing the raw row count, duplicates removed, final count, and an analysis of repeated questions, so the summary in 11.1 is based on verified output.

In [32]:
import pandas as pd

raw = pd.read_csv(f"{path}/medquad.csv").rename(
    columns={"question": "Question", "answer": "Answer"})
print("raw rows:", len(raw))
print("empty question:", (raw.Question.isna() | (raw.Question.astype(str).str.strip() == "")).sum())
print("empty answer:", (raw.Answer.isna() | (raw.Answer.astype(str).str.strip() == "")).sum())

dup = raw.duplicated(subset=["Question", "Answer"], keep="first")
print("exact duplicates removed:", dup.sum())
clean = raw[~dup].reset_index(drop=True)
print("final rows:", len(clean))

clean["q_key"] = clean.Question.str.strip().str.lower()
counts = clean.groupby("q_key").size()
print("unique questions:", len(counts))
print("questions appearing more than once:", (counts > 1).sum())
print("extra pairs:", len(clean) - len(counts))

rep = clean[clean.q_key.isin(counts[counts > 1].index)]
g = rep.groupby("q_key").agg(n_sources=("source", "nunique"),
                             n_focus=("focus_area", "nunique"),
                             n_answers=("Answer", "nunique"))
print("repeated questions in >1 source:", (g.n_sources > 1).sum())
print("repeated questions in >1 focus area:", (g.n_focus > 1).sum())
print("repeated questions with >1 distinct answer:", (g.n_answers > 1).sum())
print(rep.source.value_counts())
print(rep.sort_values("q_key")[["Question", "source", "focus_area"]].head(10))

raw rows: 16412
empty question: 0
empty answer: 5
exact duplicates removed: 48
final rows: 16364
unique questions: 14349
questions appearing more than once: 1397
extra pairs: 2015
repeated questions in >1 source: 1024
repeated questions in >1 focus area: 612
repeated questions with >1 distinct answer: 1396
source
GARD                 755
GHR                  744
NIHSeniorHealth      686
NIDDK                600
NINDS                202
MPlusHealthTopics    161
CancerGov            125
NHLBI                116
CDC                   23
Name: count, dtype: int64
                                                Question source  \
15710  How many people are affected by Alagille Syndr...  NIDDK   
9806   How many people are affected by Alagille syndr...    GHR   
15359   How many people are affected by Celiac Disease ?  NIDDK   
11721   How many people are affected by celiac disease ?    GHR   
12501  How many people are affected by cyclic vomitin...    GHR   
16343  How many people are affec

### 11.7 Checking for Empty Answers in the Final Dataset
The raw file contains 5 rows with empty answers (11.6). This cell checks whether any remain after duplicate removal, since the inclusion criterion requires a non-empty answer.

In [33]:
empty_mask = clean.Answer.isna() | (clean.Answer.astype(str).str.strip() == "")
print("empty answers in raw:", (raw.Answer.isna() | (raw.Answer.astype(str).str.strip() == "")).sum())
print("empty answers still in final dataset:", empty_mask.sum())
print(clean[empty_mask][["Question", "source", "focus_area"]])

empty answers in raw: 5
empty answers still in final dataset: 5
                                               Question source  \
3591                     What is (are) HELLP syndrome ?   GARD   
3840  What is (are) X-linked lymphoproliferative syn...   GARD   
4200            What is (are) Familial HDL deficiency ?   GARD   
4433  What is (are) Emery-Dreifuss muscular dystroph...   GARD   
6693  What is (are) Emery-Dreifuss muscular dystroph...   GARD   

                                            focus_area  
3591                                    HELLP syndrome  
3840             X-linked lymphoproliferative syndrome  
4200                           Familial HDL deficiency  
4433       Emery-Dreifuss muscular dystrophy, X-linked  
6693  Emery-Dreifuss muscular dystrophy, dominant type  


### 11.8 Building and Freezing the Final Dataset (v1)
Final dataset: exact duplicate (Question, Answer) pairs removed, then rows with empty answers excluded. A stable record_id is added and no split column is included. This supersedes the file frozen in 11.2.

In [34]:
import hashlib, os

FREEZE_DIR = "/content/drive/MyDrive/MedicalAI_Revision/frozen_v1"
os.makedirs(FREEZE_DIR, exist_ok=True)

raw = pd.read_csv(f"{path}/medquad.csv").rename(
    columns={"question": "Question", "answer": "Answer"})

dup_mask = raw.duplicated(subset=["Question", "Answer"], keep="first")
step1 = raw[~dup_mask]
empty_mask = step1.Answer.isna() | (step1.Answer.astype(str).str.strip() == "")
final = step1[~empty_mask].reset_index(drop=True)
final.insert(0, "record_id", range(len(final)))

print("raw rows:", len(raw))
print("exact duplicates removed:", dup_mask.sum())
print("empty answers removed:", empty_mask.sum())
print("final rows:", len(final))
print("unique questions (stripped, lowercased):",
      final.Question.str.strip().str.lower().nunique())
print("columns:", final.columns.tolist())
assert len(final) == len(raw) - dup_mask.sum() - empty_mask.sum()

final_path = f"{FREEZE_DIR}/medquad_final_v1.csv"
final.to_csv(final_path, index=False)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

final_hash = sha256(final_path)
print("SHA-256:", final_hash)

raw rows: 16412
exact duplicates removed: 48
empty answers removed: 5
final rows: 16359
unique questions (stripped, lowercased): 14345
columns: ['record_id', 'Question', 'Answer', 'source', 'focus_area']
SHA-256: c4fe3039215857079bfa815ebe68ce3a71e785c1781c91d8946fb52462376094


### 11.9 Creating Leak-Free Train / Validation / Test Splits
Splits are grouped by normalised question (stripped, lowercased), 70/15/15, seed 3407, so no question appears in more than one split. This supersedes the row-level split in 11.3.

In [35]:
import numpy as np

SEED = 3407
final["q_key"] = final.Question.str.strip().str.lower()

keys = np.array(sorted(final.q_key.unique()))   # sorted first so the shuffle is reproducible
rng = np.random.default_rng(SEED)
rng.shuffle(keys)

n = len(keys)
n_train, n_val = int(0.70 * n), int(0.15 * n)
train_keys = set(keys[:n_train])
val_keys = set(keys[n_train:n_train + n_val])
test_keys = set(keys[n_train + n_val:])

final["split"] = np.where(final.q_key.isin(train_keys), "train",
                  np.where(final.q_key.isin(val_keys), "val", "test"))

# --- leakage checks ---
assert train_keys.isdisjoint(val_keys)
assert train_keys.isdisjoint(test_keys)
assert val_keys.isdisjoint(test_keys)
assert (final.groupby("q_key").split.nunique() == 1).all()
ids = {s: set(final.loc[final.split == s, "record_id"]) for s in ["train", "val", "test"]}
assert ids["train"].isdisjoint(ids["val"])
assert ids["train"].isdisjoint(ids["test"])
assert ids["val"].isdisjoint(ids["test"])
assert sum(len(v) for v in ids.values()) == len(final)
print("All checks passed: no question or record appears in more than one split.")

print(final.split.value_counts())
print("unique questions per split:", final.groupby("split").q_key.nunique().to_dict())

# --- save split files and ID files ---
split_hashes = {}
for s in ["train", "val", "test"]:
    part = final[final.split == s]
    part[["record_id"]].to_csv(f"{FREEZE_DIR}/{s}_ids.csv", index=False)
    part.drop(columns=["q_key", "split"]).to_csv(f"{FREEZE_DIR}/{s}.csv", index=False)
    split_hashes[s] = sha256(f"{FREEZE_DIR}/{s}.csv")
    print(s, "SHA-256:", split_hashes[s])

All checks passed: no question or record appears in more than one split.
split
train    11404
test      2479
val       2476
Name: count, dtype: int64
unique questions per split: {'test': 2153, 'train': 10041, 'val': 2151}
train SHA-256: a0fd2e7fb3313b94c54574d4419b2e257054c5a5c64b39b6b650a03df9b16c24
val SHA-256: 0e6cbc3195a9914ca16a5870ebfabb8c58e1c58ab6f7636a0ec67bfec1655076
test SHA-256: be8bc0c806749d9e6bfbfdc7c0039896a62626de768fc106d88411c580e53552


### 11.10 Saving the Cleaning Script and Verifying Reproducibility
The cleaning and splitting steps are saved as a standalone script and re-run from the raw file; output hashes are compared with the frozen files from 11.8 and 11.9. This supersedes the script in 11.5.

In [36]:
import subprocess, shutil

cleaning_script = '''"""
MedQuAD cleaning and splitting script (v1)
Source: Kaggle pythonafroz/medquad-medical-question-answer-for-ai-research
Steps: rename columns -> drop exact duplicate (Question, Answer) pairs
       -> drop rows with empty answers -> add record_id
       -> split by normalised question (strip + lowercase), 70/15/15, seed 3407
Usage: python clean_medquad.py <path_to_medquad.csv> <output_dir>
"""
import sys, os
import numpy as np
import pandas as pd

SEED = 3407

def main(raw_path, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    raw = pd.read_csv(raw_path).rename(columns={"question": "Question", "answer": "Answer"})

    dup_mask = raw.duplicated(subset=["Question", "Answer"], keep="first")
    step1 = raw[~dup_mask]
    empty_mask = step1.Answer.isna() | (step1.Answer.astype(str).str.strip() == "")
    final = step1[~empty_mask].reset_index(drop=True)
    final.insert(0, "record_id", range(len(final)))
    final.to_csv(os.path.join(out_dir, "medquad_final_v1.csv"), index=False)

    final["q_key"] = final.Question.str.strip().str.lower()
    keys = np.array(sorted(final.q_key.unique()))
    rng = np.random.default_rng(SEED)
    rng.shuffle(keys)
    n = len(keys)
    n_train, n_val = int(0.70 * n), int(0.15 * n)
    train_keys = set(keys[:n_train])
    val_keys = set(keys[n_train:n_train + n_val])
    final["split"] = np.where(final.q_key.isin(train_keys), "train",
                      np.where(final.q_key.isin(val_keys), "val", "test"))
    for s in ["train", "val", "test"]:
        part = final[final.split == s]
        part[["record_id"]].to_csv(os.path.join(out_dir, f"{s}_ids.csv"), index=False)
        part.drop(columns=["q_key", "split"]).to_csv(os.path.join(out_dir, f"{s}.csv"), index=False)

if __name__ == "__main__":
    main(sys.argv[1], sys.argv[2])
'''

script_path = f"{FREEZE_DIR}/clean_medquad.py"
with open(script_path, "w") as f:
    f.write(cleaning_script)

repro_dir = "/content/repro_check"
shutil.rmtree(repro_dir, ignore_errors=True)
result = subprocess.run(["python", script_path, f"{path}/medquad.csv", repro_dir],
                        capture_output=True, text=True)
print(result.stdout, result.stderr)

frozen = {"medquad_final_v1.csv": final_hash,
          "train.csv": split_hashes["train"],
          "val.csv": split_hashes["val"],
          "test.csv": split_hashes["test"]}
for name, h in frozen.items():
    print(name, "MATCH" if sha256(f"{repro_dir}/{name}") == h else "MISMATCH")

 
medquad_final_v1.csv MATCH
train.csv MATCH
val.csv MATCH
test.csv MATCH


In [37]:
from google.colab import files
files.download(script_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 11.11 Dataset Manifest (Frozen Version v1)
Single record identifying the exact dataset version used for all experiments: counts, criteria, split method, seed, file hashes, cleaning-script commit, and package versions.

In [38]:
import json, datetime, platform, sys
import numpy, pandas

GIT_COMMIT = "c060ff2"
GITHUB_URL = "https://github.com/sabbir179/Multi_Agent_Medical_AI/blob/master/clean_medquad.py"
assert "PASTE" not in GIT_COMMIT and "PASTE" not in GITHUB_URL, "Fill in the commit hash and URL first"

manifest = {
    "created": datetime.datetime.now().isoformat(),
    "source": "Kaggle: pythonafroz/medquad-medical-question-answer-for-ai-research (medquad.csv)",
    "raw_rows": int(len(raw)),
    "exact_duplicates_removed": int(dup_mask.sum()),
    "empty_answers_removed": int(empty_mask.sum()),
    "final_rows": int(len(final)),
    "unique_questions": int(final.q_key.nunique()),
    "inclusion_criteria": "all rows in medquad.csv with a non-empty question and non-empty answer",
    "exclusion_criteria": "exact duplicate (Question, Answer) pairs (case-sensitive, first kept); rows with empty answers",
    "split_method": "grouped by normalised question (strip + lowercase), 70/15/15",
    "split_seed": SEED,
    "split_rows": {k: int(v) for k, v in final.split.value_counts().items()},
    "split_unique_questions": {k: int(v) for k, v in final.groupby("split").q_key.nunique().items()},
    "sha256": {"medquad_final_v1.csv": final_hash,
               **{f"{s}.csv": h for s, h in split_hashes.items()}},
    "cleaning_script": {"file": "clean_medquad.py",
                        "sha256": sha256(script_path),
                        "github_commit": GIT_COMMIT,
                        "url": GITHUB_URL},
    "environment": {"python": sys.version.split()[0],
                    "pandas": pandas.__version__,
                    "numpy": numpy.__version__,
                    "platform": platform.platform()},
}

with open(f"{FREEZE_DIR}/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2))

{
  "created": "2026-11-23T20:58:36.152717",
  "source": "Kaggle: pythonafroz/medquad-medical-question-answer-for-ai-research (medquad.csv)",
  "raw_rows": 16412,
  "exact_duplicates_removed": 48,
  "empty_answers_removed": 5,
  "final_rows": 16359,
  "unique_questions": 14345,
  "inclusion_criteria": "all rows in medquad.csv with a non-empty question and non-empty answer",
  "exclusion_criteria": "exact duplicate (Question, Answer) pairs (case-sensitive, first kept); rows with empty answers",
  "split_method": "grouped by normalised question (strip + lowercase), 70/15/15",
  "split_seed": 3407,
  "split_rows": {
    "train": 11404,
    "test": 2479,
    "val": 2476
  },
  "split_unique_questions": {
    "test": 2153,
    "train": 10041,
    "val": 2151
  },
  "sha256": {
    "medquad_final_v1.csv": "c4fe3039215857079bfa815ebe68ce3a71e785c1781c91d8946fb52462376094",
    "train.csv": "a0fd2e7fb3313b94c54574d4419b2e257054c5a5c64b39b6b650a03df9b16c24",
    "val.csv": "0e6cbc3195a9914ca16a

In [39]:
counts = final.groupby("q_key").size()
rep = final[final.q_key.isin(counts[counts > 1].index)]
g = rep.groupby("q_key").agg(n_sources=("source", "nunique"),
                             n_answers=("Answer", "nunique"))
print("repeated questions:", (counts > 1).sum())
print("extra pairs:", len(final) - len(counts))
print("repeated in >1 source:", (g.n_sources > 1).sum())
print("repeated with >1 distinct answer:", (g.n_answers > 1).sum())

repeated questions: 1396
extra pairs: 2014
repeated in >1 source: 1023
repeated with >1 distinct answer: 1396


## 12. DeepSeek Retraining

Fine-tuning DeepSeek-R1-Distill-Llama-8B on the frozen MedQuAD dataset,
using the original hyperparameters documented in the training script,
adapted to use the verified dataset and this environment. Training
time, GPU memory usage, and inference latency are logged throughout,
as requested.

### 12.1 Installing Required Packages

In [31]:
!pip install unsloth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.0/32.0 MB 75.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 155.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 55.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 57.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 127.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 104.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 109.8 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [unsloth]


### 12.2 Run Setup: Environment, Package Versions and Data Check

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, platform, json, datetime, subprocess, hashlib
import torch

FREEZE_DIR = "/content/drive/MyDrive/MedicalAI_Revision/frozen_v1"
RUN_ID = "deepseek_run01"
RUN_DIR = f"{FREEZE_DIR}/runs/{RUN_ID}"
os.makedirs(RUN_DIR, exist_ok=True)

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

# 1. confirm the data files are exactly the frozen version
manifest = json.load(open(f"{FREEZE_DIR}/manifest.json"))
for name, h in manifest["sha256"].items():
    print(name, "OK" if sha256(f"{FREEZE_DIR}/{name}") == h else "MISMATCH")

# 2. record the environment
env = {
    "run_id": RUN_ID,
    "timestamp": datetime.datetime.now().isoformat(),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_total_mem_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
                        if torch.cuda.is_available() else None,
    "nvidia_smi": sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader"),
    "dataset_sha256": manifest["sha256"],
    "pip_freeze": sh("pip freeze").splitlines(),
}
with open(f"{RUN_DIR}/environment.json", "w") as f:
    json.dump(env, f, indent=2)

print("GPU:", env["gpu"], "|", env["gpu_total_mem_gb"], "GB")
for p in ["unsloth", "transformers", "trl", "peft", "bitsandbytes", "datasets"]:
    print(p, [l for l in env["pip_freeze"] if l.lower().startswith(p)])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
medquad_final_v1.csv OK
train.csv OK
val.csv OK
test.csv OK
GPU: Tesla T4 | 15.6 GB
unsloth ['unsloth==2026.11.10', 'unsloth_zoo==2026.11.10']
transformers ['transformers==5.17.0']
trl ['trl==1.15.0']
peft ['peft==0.21.2']
bitsandbytes ['bitsandbytes==0.50.2']
datasets ['datasets==4.8.5']


### 12.3 Loading the Model and the Frozen Splits
DeepSeek-R1-Distill-Llama-8B (4-bit) is loaded with the original settings. Data is loaded from the frozen train/val/test files only, and prompts use the original template, so Run 1 reproduces the original recipe on leak-free data.

In [2]:
from unsloth import FastLanguageModel
from datasets import load_dataset
import numpy as np, time

max_seq_length = 2048
t0 = time.time()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/DeepSeek-R1-Distill-Llama-8B",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)
print("model load seconds:", round(time.time() - t0))

# Original prompt template, unchanged (note: the answer goes in the second slot, inside <think>)
training_prompt_style = '''Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.
Please answer the following medical question.

### Question:
{}

### Response:
<think>
{}
</think>'''

data_files = {s: f"{FREEZE_DIR}/{s}.csv" for s in ["train", "val", "test"]}
ds = load_dataset("csv", data_files=data_files)

EOS_TOKEN = tokenizer.eos_token

def formatting_prompt_func(examples):
    texts = [training_prompt_style.format(q, a, "") + EOS_TOKEN
             for q, a in zip(examples["Question"], examples["Answer"])]
    return {"text": texts}

ds = ds.map(formatting_prompt_func, batched=True)
print({k: len(v) for k, v in ds.items()})
print(ds["train"][0]["text"][:700])

# token lengths on a 1,000-example sample, to document truncation at 2,048 tokens
sample = ds["train"].shuffle(seed=3407).select(range(1000))
lens = np.array([len(tokenizer(t).input_ids) for t in sample["text"]])
print("mean tokens:", lens.mean().round(1), "| max:", lens.max(),
      "| % over 2048:", round((lens > 2048).mean() * 100, 1))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1123 21:43:38.937000 28032 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1123 21:43:38.976000 28032 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.10: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: LlamaTokenizer for unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit did not match tokenizer.json (transformers v5 replaced its pre-tokenizer), so it was rebuilt from tokenizer.json.
Unsloth: Will load unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit as a legacy tokenizer.


model load seconds: 80


... (3 outputs left out)

Map:   0%|          | 0/11404 [00:00<?, ? examples/s]

Map:   0%|          | 0/2476 [00:00<?, ? examples/s]

Map:   0%|          | 0/2479 [00:00<?, ? examples/s]

{'train': 11404, 'val': 2476, 'test': 2479}
Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.
Please answer the following medical question.

### Question:
What are the symptoms of Glaucoma ?

### Response:
<think>
Symptoms of Glaucoma  Glaucoma can develop in one or both eyes. The most common type of glaucoma, open-angle glaucoma, has no symptoms at first. It causes no pain, and vision seems normal. Without treatment, people with glaucoma will slowly lose their peripheral, or side vision. They seem to be
mean tokens: 338.0 | max: 4204 | % over 2048: 0.7


### 12.4 LoRA Setup and Trainer Configuration (Run 1)
Hyperparameters, LoRA settings, seeds and prompt template are identical to the original script. Differences: training uses train.csv only (original used all rows), validation data is attached for post-training evaluation, and the trainer uses the TRL 1.15 SFTConfig API. Metrics are logged locally instead of to Weights & Biases.

In [3]:
import torch, json
from unsloth import is_bfloat16_supported
from trl import SFTTrainer, SFTConfig

model_lora = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

use_bf16 = is_bfloat16_supported()
sft_config = SFTConfig(
    output_dir=f"{RUN_DIR}/outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    max_steps=60,
    warmup_steps=5,
    learning_rate=2e-4,
    fp16=not use_bf16,
    bf16=use_bf16,
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    gradient_checkpointing=False,      # handled by Unsloth, as in the original script
    dataset_text_field="text",
    max_length=2048,
    dataset_num_proc=2,
    report_to="none",
    save_strategy="steps",
    save_steps=20,
    save_total_limit=3,
)

trainer = SFTTrainer(
    model=model_lora,
    processing_class=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["val"],
    args=sft_config,
)
print("trainer ready | train examples:", len(trainer.train_dataset),
      "| val examples:", len(ds["val"]))
print("precision:", "bf16" if use_bf16 else "fp16")

config = {
    "run_id": RUN_ID,
    "model": "unsloth/DeepSeek-R1-Distill-Llama-8B",
    "load_in_4bit": True,
    "max_seq_length": 2048,
    "lora": {"r": 16, "alpha": 16, "dropout": 0, "bias": "none",
             "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                                "gate_proj", "up_proj", "down_proj"],
             "use_rslora": False, "gradient_checkpointing": "unsloth",
             "random_state": 3407},
    "training": {k: getattr(sft_config, k) for k in [
        "per_device_train_batch_size", "gradient_accumulation_steps", "max_steps",
        "warmup_steps", "learning_rate", "optim", "weight_decay",
        "lr_scheduler_type", "seed", "logging_steps", "fp16", "bf16", "max_length"]},
    "examples_seen": 60 * 2 * 4,
    "data_files": {"train": "train.csv", "val": "val.csv", "test": "test.csv"},
    "template_note": "original template unchanged; answer is placed inside the <think> block",
    "truncation_note": "0.7% of sampled training examples exceed 2048 tokens",
}
with open(f"{RUN_DIR}/config.json", "w") as f:
    json.dump(config, f, indent=2, default=str)
print("config saved")

Unsloth 2026.11.10 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Unsloth: compiling each transformer block as one region (LlamaDecoderLayer).


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/11404 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2476 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
trainer ready | train examples: 11404 | val examples: 2476
precision: fp16
config saved


### 12.5 Training Run 1: Timing, GPU Memory, Validation Loss and Checkpoints
Training time, peak GPU memory, loss log and validation loss are recorded and saved to Drive together with the LoRA adapter and intermediate checkpoints.

In [4]:
import os, time, json, torch

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

t0 = time.time()
trainer_stats = trainer.train()
train_seconds = time.time() - t0

peak_allocated = torch.cuda.max_memory_allocated() / 1e9
peak_reserved = torch.cuda.max_memory_reserved() / 1e9
print(f"training time: {train_seconds:.0f} s ({train_seconds/60:.1f} min)")
print(f"peak GPU memory: {peak_allocated:.2f} GB allocated, {peak_reserved:.2f} GB reserved")

# validation loss, measured after training (no change to the training recipe)
t1 = time.time()
val_metrics = trainer.evaluate()
eval_seconds = time.time() - t1
print("validation metrics:", val_metrics)

# save the LoRA adapter and tokenizer
adapter_dir = f"{RUN_DIR}/lora_adapter"
model_lora.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

results = {
    "run_id": RUN_ID,
    "train_seconds": round(train_seconds, 1),
    "eval_seconds": round(eval_seconds, 1),
    "gpu_peak_allocated_gb": round(peak_allocated, 2),
    "gpu_peak_reserved_gb": round(peak_reserved, 2),
    "gpu_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
    "train_metrics": trainer_stats.metrics,
    "val_metrics": val_metrics,
    "log_history": trainer.state.log_history,
    "adapter_dir": adapter_dir,
    "checkpoints": sorted(os.listdir(f"{RUN_DIR}/outputs")),
}
with open(f"{RUN_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)
print("saved:", results["checkpoints"], "+ lora_adapter + results.json")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 11,404 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.005505
20,1.255465
30,1.140626
40,1.142174
50,1.063179
60,1.100018


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/MedicalAI_Revision/frozen_v1/runs/deepseek_run01/outputs/checkpoint-20/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/MedicalAI_Revision/frozen_v1/runs/deepseek_run01/outputs/checkpoint-40/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/MedicalAI_Revision/frozen_v1/runs/deepseek_run01/outputs/checkpoint-60/tokenizer_config.json.


training time: 464 s (7.7 min)
peak GPU memory: 14.08 GB allocated, 14.26 GB reserved


Training Loss,Validation Loss,Step
1.100018,0.989517,60


validation metrics: {'eval_loss': 0.989517331123352}


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/MedicalAI_Revision/frozen_v1/runs/deepseek_run01/lora_adapter/tokenizer_config.json.


saved: ['README.md', 'checkpoint-20', 'checkpoint-40', 'checkpoint-60'] + lora_adapter + results.json


### 12.6 Inference Latency and Test-Set Generations (100-question sample)
Answers are generated with greedy decoding (deterministic), max 512 new tokens, for a fixed random sample of test questions (seed 3407). Prompt format matches training. Two warm-up generations are excluded from timing.

In [5]:
import time, gc, json, numpy as np, pandas as pd, torch
from unsloth import FastLanguageModel

gc.collect()
torch.cuda.empty_cache()
FastLanguageModel.for_inference(model_lora)

# prompt = training template up to the start of the <think> block
prompt_prefix = training_prompt_style.split("<think>")[0] + "<think>\n"

N_SAMPLES = 100
MAX_NEW_TOKENS = 512
test_sample = ds["test"].shuffle(seed=3407).select(range(N_SAMPLES))

def generate_answer(question):
    inputs = tokenizer(prompt_prefix.format(question), return_tensors="pt").to("cuda")
    torch.cuda.synchronize()
    t = time.time()
    with torch.no_grad():
        out = model_lora.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                                  do_sample=False, use_cache=True)
    torch.cuda.synchronize()
    seconds = time.time() - t
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).split("</think>")[0].strip()
    return text, seconds, len(new_tokens)

for q in test_sample["Question"][:2]:      # warm-up, not counted
    generate_answer(q)

rows = []
for i, (rid, q, ref) in enumerate(zip(test_sample["record_id"],
                                      test_sample["Question"],
                                      test_sample["Answer"])):
    ans, sec, n = generate_answer(q)
    rows.append({"record_id": rid, "question": q, "reference": ref,
                 "generated": ans, "latency_s": sec, "new_tokens": n})
    if (i + 1) % 10 == 0:
        print(f"{i+1}/{N_SAMPLES} done | mean latency so far: "
              f"{np.mean([r['latency_s'] for r in rows]):.1f} s")

res = pd.DataFrame(rows)
res.to_csv(f"{RUN_DIR}/test_sample_generations.csv", index=False)

lat = res.latency_s.values
latency_stats = {
    "n_questions": N_SAMPLES,
    "decoding": "greedy",
    "max_new_tokens": MAX_NEW_TOKENS,
    "mean_s": round(float(lat.mean()), 2),
    "median_s": round(float(np.median(lat)), 2),
    "p95_s": round(float(np.percentile(lat, 95)), 2),
    "min_s": round(float(lat.min()), 2),
    "max_s": round(float(lat.max()), 2),
    "mean_new_tokens": round(float(res.new_tokens.mean()), 1),
    "tokens_per_second": round(float(res.new_tokens.sum() / lat.sum()), 1),
    "share_hitting_token_limit": round(float((res.new_tokens >= MAX_NEW_TOKENS).mean()), 3),
    "gpu": torch.cuda.get_device_name(0),
}
with open(f"{RUN_DIR}/latency.json", "w") as f:
    json.dump(latency_stats, f, indent=2)
print(json.dumps(latency_stats, indent=2))
print(res.generated.iloc[0][:500])

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

10/100 done | mean latency so far: 16.1 s


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

20/100 done | mean latency so far: 14.0 s


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

30/100 done | mean latency so far: 13.1 s


... (10 outputs left out)

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

90/100 done | mean latency so far: 12.1 s


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

100/100 done | mean latency so far: 12.5 s
{
  "n_questions": 100,
  "decoding": "greedy",
  "max_new_tokens": 512,
  "mean_s": 12.52,
  "median_s": 14.61,
  "p95_s": 19.49,
  "min_s": 1.11,
  "max_s": 19.76,
  "mean_new_tokens": 328.1,
  "tokens_per_second": 26.2,
  "share_hitting_token_limit": 0.47,
  "gpu": "Tesla T4"
}
The genetic changes related to dermatofifibrosarcoma protuberans are not well understood.  The gene responsible for the development of dermatofibrosarcoma protuberans is unknown.  The gene responsible for the development of dermatofibrosarcoma protuberans is unknown.  The gene responsible for the development of dermatofibrosarcoma protuberans is unknown.  The gene responsible for the development of dermatofibrosarcoma protuberans is unknown.  The gene responsible for the development of dermatofi


### 12.7 Text-Quality Scores and Repetition Check (100-question sample)

In [6]:
%pip install -q rouge-score sacrebleu

import re, json, numpy as np, pandas as pd
from collections import Counter
from rouge_score import rouge_scorer
import sacrebleu

res = pd.read_csv(f"{RUN_DIR}/test_sample_generations.csv")
res["generated"] = res.generated.fillna("").astype(str)

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
scores = [scorer.score(ref, gen) for ref, gen in zip(res.reference, res.generated)]
rouge = {k: round(float(np.mean([s[k].fmeasure for s in scores])), 4)
         for k in ["rouge1", "rouge2", "rougeL"]}
bleu = sacrebleu.corpus_bleu(res.generated.tolist(), [res.reference.tolist()]).score / 100

def has_repetition(text):
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if len(s.strip()) > 20]
    return bool(sents) and max(Counter(sents).values()) >= 3

res["repetitive"] = res.generated.apply(has_repetition)

quality = {
    "n_questions": len(res),
    **rouge,
    "bleu_corpus_0_to_1": round(float(bleu), 4),
    "share_repetitive_answers": round(float(res.repetitive.mean()), 3),
    "share_hitting_token_limit": round(float((res.new_tokens >= MAX_NEW_TOKENS).mean()), 3),
    "mean_words_generated": round(float(res.generated.str.split().str.len().mean()), 1),
    "mean_words_reference": round(float(res.reference.str.split().str.len().mean()), 1),
}
with open(f"{RUN_DIR}/quality_sample.json", "w") as f:
    json.dump(quality, f, indent=2)
print(json.dumps(quality, indent=2))

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
{
  "n_questions": 100,
  "rouge1": 0.279,
  "rouge2": 0.1418,
  "rougeL": 0.212,
  "bleu_corpus_0_to_1": 0.1151,
  "share_repetitive_answers": 0.22,
  "share_hitting_token_limit": 0.47,
  "mean_words_generated": 236.3,
  "mean_words_reference": 202.7
}
